# Sample-level & outlier views

Interactive driver for `analysis/profiles/plots.py`.

Select the dataset (or pooled set) with the `DATASET` env var **before launching Jupyter**, e.g.
`DATASET=alphaseq jupyter lab`, or set `base` by hand in the load cell below.
Cross-metric figures (clustermap / filter / profiles) use the standardized+aligned table;
`per_dataset` and `outliers` use the raw values.

In [ ]:
%load_ext autoreload
%autoreload 2
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

from analysis.distributions.wide import add_binder_class
from config.paths import RAW_DATA_DIR
from config.datasets import cfg
from preprocessing.metric_meta import load_reference_values, load_families, get_family
from analysis.distributions.profiles import (
    plot_sample_clustermap, plot_filter_heatmap, plot_sample_profiles,
    plot_per_dataset, plot_outliers, outlier_table, cluster_association,
)
# Uniform figure styling: one validated palette and recessive axes/grid for every figure
# in the project (config/palette.py). Call once, before plotting.
from config.palette import apply_plot_style
apply_plot_style()


In [ ]:
# pick the dataset directory 
base = RAW_DATA_DIR / cfg.name               # cluster: set DATASET=... before launch
# base = Path("data/raw/alphaseq")           # local override, if not using config paths

raw = add_binder_class(pd.read_csv(base / "merged.csv"))
sa = base / "merged_scaled_aligned.csv"
if not sa.exists():                          # scaling must be done up front (run_scale.py)
    raise FileNotFoundError(f"{sa} not found - run run_scale.py first; profiles expect a scaled+aligned table.")
scaled = add_binder_class(pd.read_csv(sa))
print(base.name, "| samples:", len(raw), "| classes:", raw["binder_class"].value_counts().to_dict())

## 1. Sample × metric clustermap  (standardized, aligned)

In [ ]:
plot_sample_clustermap(scaled);

## 2. Boolean filter matrix  (per-metric median = placeholder cutoff)

In [ ]:
# plot_filter_heatmap takes {family: value}; the reference values live per metric
_ref, _fams = load_reference_values(), load_families()
thr = {get_family(m, _fams): v for m, v in _ref.items() if get_family(m, _fams)}
fig, report = plot_filter_heatmap(raw, thr)                 # binary pass/fail
print("no threshold (skipped):", report["no_threshold"])
print("unmatched columns (dropped):", report["unmatched"])
plot_filter_heatmap(raw, thr, mode="graded")           # graded: margin from threshold (blue=pass, red=fail)

## 3. Selected-sample profiles vs group bands

In [ ]:
plot_sample_profiles(scaled, n=3);

## 4. Per-dataset distribution  (raw values; facets on `dataset` else `source`)

In [ ]:
plot_per_dataset(raw, top=4);

## 5. Outlier / scale inspection

Change `scale` to `error` / `likelihood` / `distance` / `ratio` to inspect the other families.
The table gives the robust (median/IQR) outlier flag counts; the plot shows whether the
spread is cross-model **scale** or true outliers.

In [ ]:
display(outlier_table(raw, scale="energy"))
plot_outliers(raw, scale="energy");

## 6. Cluster association

Does whole-profile sample clustering line up with `binder_class` / `dataset`? 
ARI ~ 0 means no; `block` lists the k=2 split-off samples.

In [ ]:
ari, block = cluster_association(scaled)
display(ari)
print("split-off block:", len(block))
display(block)